<a href="https://colab.research.google.com/github/Chathuranga630/sionna/blob/main/Resilience_Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ============================================================
# CLEAN SIONNA 2.1 INSTALLATION FOR GOOGLE COLAB
# ============================================================

!pip install -q --upgrade pip

# Sionna 2.1 / Numba-compatible NumPy
!pip install -q --force-reinstall "numpy==2.2.6"

# Sionna 2.1
!pip install -q "sionna==2.1.0"

print("Installation finished.")
print("NOW RESTART THE RUNTIME:")
print("Runtime -> Restart session")

Installation finished.
NOW RESTART THE RUNTIME:
Runtime -> Restart session


In [1]:
import numpy as np

print("NumPy version:", np.__version__)

import numba

print("Numba version:", numba.__version__)

NumPy version: 2.2.6
Numba version: 0.61.2


In [2]:
import sionna.phy

print("Sionna PHY imported successfully")
print("Sionna version:", getattr(sionna.phy, "__version__", "2.x"))

Sionna PHY imported successfully
Sionna version: 2.x


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [6]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

import sionna.phy
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.phy.channel import AWGN, OFDMChannel
from sionna.phy.channel.tr38901 import AntennaArray, UMi
from sionna.phy.utils import compute_ber, ebnodb2no

print("NumPy :", np.__version__)
print("PyTorch:", torch.__version__)
print("Sionna imported successfully")

NumPy : 2.2.6
PyTorch: 2.11.0+cu128
Sionna imported successfully


In [7]:
# ---------------- Simulation assumptions ----------------
N_USERS = 4
PRB = 4
EBNO_DB = np.arange(-2, 16, 2)

# Service/QoS assumptions (replace for your target service)
TARGET_RATE_MBPS = np.array([20., 10., 5., 5.])  # priority-weighted service targets
PRIORITY = np.array([1.0, 0.8, 0.5, 0.5])
Q_THRESHOLD = 0.80
ETH = 0.75                     # normalized QoS-deviation threshold (assumption)
T_NMIN_MS = 10.0               # I1 threshold (assumption)
DMIN_MS = 8.0                   # I2 threshold (assumption)
TPROC_MS = 7.0                 # I4 threshold (assumption)
TSLEEP_MAX_MS = 20.0            # I5 threshold (assumption)

# Prompt/control timing assumptions
TGEN_MS = np.array([1.5,1.7,1.6,1.8,1.9,2.0,2.1,2.0,2.2,2.3])
D_MS = np.array([0.8,0.9,1.0,1.1,1.2,1.4,1.3,1.5,1.6,1.8])
C_MS = 0.5

# RIS timing assumptions; physical channel effect is applied later as a gain parameter
T_RIS_IT_MS = np.array([0.6,0.7,0.5,0.8,0.9,0.7,0.8,1.0,0.9,1.1])
RIS_GAIN_DB = 3.0

In [8]:
# A compact PUSCH configuration
pusch_config = PUSCHConfig()
pusch_config.n_size_bwp = PRB
pusch_config.num_antenna_ports = 1
pusch_config.num_layers = 1
pusch_config.dmrs.dmrs_port_set = [0]

tx = PUSCHTransmitter(pusch_config)
rx = PUSCHReceiver(tx)

# For a fast Colab run, use AWGN first. Replace with the UMi block below
# when you want a 3GPP spatial channel.
channel_awgn = AWGN()

def run_awgn_ber(ebno_db_values, batch_size=32):
    ber = []
    for ebno in ebno_db_values:
        # The exact no conversion depends on the configured waveform;
        # this is a compact link-level example for the resilience workflow.
        x, b = tx(batch_size)
        no = ebnodb2no(ebno, pusch_config)
        y = channel_awgn(x, no)
        b_hat = rx(y, no)
        ber.append(float(compute_ber(b, b_hat).numpy()))
    return np.array(ber)

ber = run_awgn_ber(EBNO_DB)
pd.DataFrame({"Eb/N0_dB":EBNO_DB, "BER":ber})

TypeError: ebnodb2no() missing 1 required positional argument: 'coderate'